In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [4]:
df = pd.read_csv("/content/drive/MyDrive/DATASET/Phishing_Email.csv")


df=df.rename(columns={'Email Text':'email_text','Email Type':'label'})
print(df.columns)
df['email_text'] = df['email_text'].astype(str).str.strip()

df['label'] = np.where(df['label']=='Safe Email',0,1)

train_texts, temp_texts, train_labels, temp_labels = train_test_split(df['email_text'], df['label'], test_size=0.3, random_state=42)
val_texts, test_texts, val_labels, test_labels = train_test_split(temp_texts, temp_labels, test_size=0.5, random_state=42)


Index(['Unnamed: 0', 'email_text', 'label'], dtype='object')


In [5]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
import torch

class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} | {"labels": torch.tensor(self.labels[idx])}

    def __len__(self):
        return len(self.labels)

train_dataset = EmailDataset(train_encodings, list(train_labels))
val_dataset = EmailDataset(val_encodings, list(val_labels))
test_dataset = EmailDataset(test_encodings, list(test_labels))


In [8]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
!pip uninstall wandb

In [10]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    logging_steps=10,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

)

trainer.train()


Step,Training Loss
10,0.665528
20,0.621196
30,0.525915
40,0.500720
50,0.316396
60,0.312888
70,0.182738
80,0.177533
90,0.195034
100,0.218466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3264, training_loss=0.05632320567326365, metrics={'train_runtime': 2506.9113, 'train_samples_per_second': 20.83, 'train_steps_per_second': 1.302, 'total_flos': 6917447557816320.0, 'train_loss': 0.05632320567326365, 'epoch': 4.0})

In [11]:
trainer.evaluate(test_dataset)
preds = trainer.predict(test_dataset)
pred_labels = preds.predictions.argmax(-1)
from sklearn.metrics import classification_report

print(classification_report(test_labels, pred_labels, target_names=["Safe", "Phishing"]))


              precision    recall  f1-score   support

        Safe       0.99      0.97      0.98      1687
    Phishing       0.96      0.98      0.97      1111

    accuracy                           0.98      2798
   macro avg       0.97      0.98      0.98      2798
weighted avg       0.98      0.98      0.98      2798



In [12]:
model.save_pretrained("/content/drive/MyDrive/DATASET/phishing-model")
tokenizer.save_pretrained("/content/drive/MyDrive/DATASET/phishing-model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/DATASET/phishing-model/tokenizer_config.json',
 '/content/drive/MyDrive/DATASET/phishing-model/tokenizer.json')